# 02 — 状態・入力・座標系

## 目的
次元が合うだけの誤接続を防ぐ。`x(24)`, `u(24)`, `rbdState(36)` は同じ24/36個の
「数」ではなく、順序・単位・frameを含む契約である。

実装対応:
- `legged_controllers/config/a1/task.info`
- `ocs2_legged_robot` の centroidal model helpers
- [`StateEstimateBase.cpp`](https://github.com/qiayuanliao/legged_control/tree/a7f381c0367e98e31c01336e678eef47e304d40d/legged_estimation/src/StateEstimateBase.cpp)


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists():
        ROOT = candidate
        break

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
print("repository:", ROOT)


repository: /home/takuya/work/mpc_dog


## 主要ベクトル

\[
x=[v_{\mathrm{com}}(3),\,L/m(3),\,p_b(3),\,(\psi,\theta,\phi)(3),\,q_j(12)]
\]
\[
u=[f_c(12),\,v_j(12)]
\]

`rbdState(36)`:
ZYX(3), base position(3), joint angles(12), world angular velocity(3),
world linear velocity(3), joint velocities(12)。

- world = odom側、base = 胴体固定側
- 姿勢は ZYX の **格納順 yaw, pitch, roll**
- 関節順と接触脚順はコメント上で差がある箇所があるため、名前を正本にする


In [2]:
# 明示的なsliceで契約をコード化する。
x = np.zeros(24)
blocks_x = {
    "v_com": slice(0, 3), "L_over_m": slice(3, 6),
    "base_position": slice(6, 9), "zyx": slice(9, 12),
    "joint_angles": slice(12, 24),
}
x[blocks_x["base_position"]] = [1.0, 2.0, 0.30]
x[blocks_x["zyx"]] = [np.deg2rad(30), 0.0, 0.0]
assert all(x[s].shape == (3,) for k, s in blocks_x.items() if k != "joint_angles")
assert x[blocks_x["joint_angles"]].shape == (12,)
x


array([0.    , 0.    , 0.    , 0.    , 0.    , 0.    , 1.    , 2.    ,
       0.3   , 0.5236, 0.    , 0.    , 0.    , 0.    , 0.    , 0.    ,
       0.    , 0.    , 0.    , 0.    , 0.    , 0.    , 0.    , 0.    ])

In [3]:
# body前方速度をworldへ回す。yaw=30 degならx/y双方に成分が出る。
def Rz(yaw):
    c, s = np.cos(yaw), np.sin(yaw)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

v_body = np.array([0.5, 0.0, 0.0])
yaw = x[9]
v_world = Rz(yaw) @ v_body
print("v_body [m/s]:", v_body)
print("v_world [m/s]:", v_world)
assert np.allclose(np.linalg.norm(v_world), np.linalg.norm(v_body))


v_body [m/s]: [0.5 0.  0. ]
v_world [m/s]: [0.433 0.25  0.   ]


## よくある誤り
- `x[:3]`をbase位置だと思う（実際は正規化並進運動量/CoM速度）。
- `u[12:]`をトルクだと思う（実際は関節速度）。
- local IMU角速度とworld角速度を混ぜる。
- `LF,LH,RF,RH` と接触名の順を無検証で同じとみなす。

### 演習
すべての境界に `shape`, `unit`, `frame`, `leg order`, `rate` の5項目を書く。
これが書けない配列は、数式変更より先に調査する。


## 章固有の背景
                同じ長さのvectorでも、順序・frame・単位が違えば物理的には別の型である。

                ## 章固有の目的
                24状態、24入力、36 rigid-body stateのsliceを、変換symbolと数式に結び付ける。

                ## この章のASCIIデータフロー
                ```text
                rbdState(36: ZYX,p,q,omega,v,dq) -> CentroidalModelRbdConversions
                                            -> x(24: h/m,pose,q)
u(24: four world forces,dq*) -----------------> centroidal dynamics
                ```

                ## 上流C++ / faithful pseudocode と数式の行対応
                ```cpp
                // external/legged_control/legged_estimation/src/StateEstimateBase.cpp
rbdState.segment<3>(0) = zyx;       // [yaw,pitch,roll]
rbdState.segment<3>(3) = position;  // p_b^W
rbdState.segment<12>(6) = q;        // q_j
rbdState.segment<3>(18) = omegaW;   // omega_b^W
rbdState.segment<3>(21) = vW;       // v_b^W
rbdState.segment<12>(24) = dq;      // dq_j
// ocs2 centroidal helper contract from config/a1/task.info
x = [h_linear/m, h_angular/m, p_b, ZYX, q_j];
u = [f_LF^W,f_RF^W,f_LH^W,f_RH^W,dq_j];
                ```

                **事実のラベル**: `external/legged_control/` の記述はcommit
                `a7f381c0367e98e31c01336e678eef47e304d40d` の上流実装事実。数式展開はそのinterfaceを説明する理論。
                `src/legged_control_mujoco/` に言及した行はproject所有adapterの実装であり、
                ROS1/OCS2 SQP原実装とは同一ではない。

                ## 章固有の結論
                `x[:6]` は正規化centroidal momentum、`u[:12]` はworld contact force、`u[12:]` はjoint velocityであり、torqueはWBC後まで現れない。
